# Radiology Report Generation: Training on Kaggle GPU

This notebook trains the multimodal Vision-Language Model (**ResNet CNN Encoder + Transformer Decoder**) for Chest X-Ray findings generation.

### Workflow Overview:
1. **Accelerated Training**: Runs on Kaggle GPU (T4 x2 or P100).
2. **Auto-dataset discovery**: Locates `annotation.json` and `images/` from any attached IU X-Ray dataset.
3. **Regularized Optimization**: Uses differential learning rates (small LR for pretrained ResNet backbone, higher LR for Transformer decoder), AdamW with weight decay, label smoothing, and dropout to prevent overfitting.
4. **Evaluation**: Evaluates the best checkpoint on the test split (`BLEU-1`, `BLEU-4`, `ROUGE-L`).
5. **Export**: Packages `best_model.pt` and `vocab.json` into a zip file for download to your local machine.

In [ ]:
# 1. Environment & GPU Check
import os
import sys
import json
import math
import re
import glob
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("WARNING: CUDA not detected! Running on CPU (Go to Settings -> Accelerator -> GPU T4 x2)")
    device = torch.device("cpu")


## 2. Dataset Discovery
We search `/kaggle/input` to auto-detect the annotation JSON file and the image directory.

In [ ]:
# 2. Smart Dataset Path Auto-Detection
input_dir = "/kaggle/input"
annotation_path = None
image_dir = None

# Look for annotation.json or similar containing train/val/test splits
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith(".json") and any(k in f.lower() for k in ["annotation", "iu", "r2gen", "dataset"]):
            cand = os.path.join(root, f)
            try:
                with open(cand, "r") as jf:
                    data = json.load(jf)
                    if isinstance(data, dict) and "train" in data:
                        annotation_path = cand
                        print(f"Found valid annotation file: {annotation_path}")
                        break
            except Exception:
                pass
    if annotation_path:
        break

# Look for directory with images
for root, dirs, files in os.walk(input_dir):
    if "images" in os.path.basename(root).lower():
        image_dir = root
        print(f"Found images directory: {image_dir}")
        break
    png_or_jpg = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    if len(png_or_jpg) > 10:
        image_dir = root
        print(f"Found images directory (by image count): {image_dir}")
        break

# Fallback if not automatically located:
if not annotation_path or not image_dir:
    print("\nAvailable paths in /kaggle/input:")
    for p in glob.glob("/kaggle/input/**", recursive=True)[:30]:
        print(" ", p)
    annotation_path = annotation_path or "/kaggle/input/iu-xray/annotation.json"
    image_dir = image_dir or "/kaggle/input/iu-xray/images"

print(f"\nUsing annotation path: {annotation_path}")
print(f"Using images directory: {image_dir}")

if os.path.exists(annotation_path):
    with open(annotation_path) as f:
        ann = json.load(f)
    print("\nDataset splits summary:")
    for split in ["train", "val", "test"]:
        if split in ann:
            print(f"  Split '{split}': {len(ann[split])} samples")


## 3. Vocabulary Builder
Builds a word-level vocabulary strictly from the **training split** reports.

In [ ]:
# 3. Vocabulary
PAD_TOKEN = "<pad>"
START_TOKEN = "<start>"
END_TOKEN = "<end>"
UNK_TOKEN = "<unk>"

class Vocabulary:
    def __init__(self, min_freq: int = 3):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}

    @staticmethod
    def tokenize(text: str):
        text = text.lower()
        text = re.sub(r"[^a-z0-9.,\s]", " ", text)
        return text.split()

    def build(self, reports):
        """reports: iterable of raw report strings (training split only!)."""
        counter = Counter()
        for r in reports:
            counter.update(self.tokenize(r))

        specials = [PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN]
        words = specials + sorted(w for w, c in counter.items() if c >= self.min_freq)
        self.word2idx = {w: i for i, w in enumerate(words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        return self

    def encode(self, text: str, max_len: int = None):
        tokens = [START_TOKEN] + self.tokenize(text) + [END_TOKEN]
        unk_idx = self.word2idx[UNK_TOKEN]
        ids = [self.word2idx.get(t, unk_idx) for t in tokens]
        if max_len is not None:
            ids = ids[:max_len]
            ids = ids + [self.word2idx[PAD_TOKEN]] * (max_len - len(ids))
        return ids

    def decode(self, ids, stop_at_end: bool = True):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK_TOKEN)
            if w in (START_TOKEN, PAD_TOKEN):
                continue
            if w == END_TOKEN:
                if stop_at_end:
                    break
                continue
            words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump(self.word2idx, f)

    def load(self, path: str):
        with open(path) as f:
            self.word2idx = json.load(f)
        self.idx2word = {int(i): w for w, i in self.word2idx.items()}
        return self


## 4. Dataset Loader & Preprocessing
Loads frontal + lateral views per study, pads single views, and normalizes using ImageNet mean/std.

In [ ]:
# 4. Dataset & Augmentation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def build_transform(image_size: int = 224, train: bool = True):
    # Note: Horizontal flip is omitted for chest X-rays to preserve laterality
    ops = [transforms.Resize((image_size, image_size))]
    if train:
        ops.append(transforms.ColorJitter(brightness=0.1, contrast=0.1))
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

class IUXrayDataset(Dataset):
    def __init__(self, annotation_path, image_dir, vocab, split="train",
                 image_size=224, max_len=100, max_images=2):
        with open(annotation_path) as f:
            ann = json.load(f)
        if split not in ann:
            raise ValueError(f"Split '{split}' not in annotation file (keys: {list(ann.keys())})")

        self.samples = ann[split]
        self.image_dir = image_dir
        self.vocab = vocab
        self.max_len = max_len
        self.max_images = max_images
        self.transform = build_transform(image_size, train=(split == "train"))

    def __len__(self):
        return len(self.samples)

    def _resolve_img_path(self, rel_path):
        cand1 = os.path.join(self.image_dir, rel_path)
        if os.path.exists(cand1):
            return cand1
        cand2 = os.path.join(self.image_dir, os.path.basename(rel_path))
        if os.path.exists(cand2):
            return cand2
        if os.path.exists(rel_path):
            return rel_path
        return cand1

    def __getitem__(self, idx):
        item = self.samples[idx]
        image_paths = item["image_path"][: self.max_images]

        images = []
        for p in image_paths:
            img_path = self._resolve_img_path(p)
            img = Image.open(img_path).convert("RGB")
            images.append(self.transform(img))

        # Duplicate last view if study only has 1 view
        while len(images) < self.max_images:
            images.append(images[-1].clone())
        images = torch.stack(images, dim=0)  # (max_images, 3, H, W)

        report_ids = self.vocab.encode(item["report"], max_len=self.max_len)
        report_tensor = torch.tensor(report_ids, dtype=torch.long)

        return images, report_tensor


## 5. Model Architecture
- **CNNEncoder**: ResNet50 backbone extracting spatial feature maps, projected to `d_model`.
- **PositionalEncoding**: Sinusoidal positional embeddings for sequence tokens.
- **ReportDecoder**: Causal autoregressive Transformer Decoder cross-attending to visual memory tokens.
- **ReportGenerator**: End-to-end wrapper supporting configurable dropout, training, and greedy generation.

In [ ]:
# 5. Model Architecture
class CNNEncoder(nn.Module):
    def __init__(self, d_model: int = 256, backbone: str = "resnet50", pretrained: bool = True):
        super().__init__()
        if backbone == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            net = models.resnet50(weights=weights)
            feat_dim = 2048
        elif backbone == "resnet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            net = models.resnet18(weights=weights)
            feat_dim = 512
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # Keep spatial feature maps (remove avgpool & fc)
        self.backbone = nn.Sequential(*list(net.children())[:-2])
        self.project = nn.Conv2d(feat_dim, d_model, kernel_size=1)

    def forward(self, images):
        """
        images: (B, V, 3, H, W) - V views per study
        returns: (B, V * h' * w', d_model) visual memory sequence
        """
        b, v, c, h, w = images.shape
        images = images.view(b * v, c, h, w)
        feats = self.backbone(images)                      # (B*V, feat_dim, h', w')
        feats = self.project(feats)                         # (B*V, d_model, h', w')
        _, d, fh, fw = feats.shape
        feats = feats.view(b, v, d, fh, fw)
        feats = feats.permute(0, 1, 3, 4, 2).reshape(b, v * fh * fw, d)
        return feats


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class ReportDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024,
                 dropout: float = 0.2, max_len: int = 100, pad_idx: int = 0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

    @staticmethod
    def _causal_mask(size: int, device):
        return torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, tgt_ids, memory):
        tgt_mask = self._causal_mask(tgt_ids.size(1), tgt_ids.device)
        pad_mask = tgt_ids == self.pad_idx
        x = self.embedding(tgt_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        out = self.decoder(tgt=x, memory=memory, tgt_mask=tgt_mask, tgt_key_padding_mask=pad_mask)
        return self.output_proj(out)


class ReportGenerator(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, backbone: str = "resnet50",
                 pretrained: bool = True, nhead: int = 8, num_layers: int = 4,
                 dropout: float = 0.2, max_len: int = 100, pad_idx: int = 0):
        super().__init__()
        self.encoder = CNNEncoder(d_model=d_model, backbone=backbone, pretrained=pretrained)
        self.decoder = ReportDecoder(vocab_size, d_model=d_model, nhead=nhead,
                                      num_layers=num_layers, dropout=dropout,
                                      max_len=max_len, pad_idx=pad_idx)

    def forward(self, images, tgt_ids):
        memory = self.encoder(images)
        return self.decoder(tgt_ids, memory)

    @torch.no_grad()
    def generate(self, images, vocab, max_len: int = 100, device="cpu"):
        self.eval()
        memory = self.encoder(images.to(device))
        b = images.size(0)
        start_idx = vocab.word2idx[START_TOKEN]
        end_idx = vocab.word2idx[END_TOKEN]

        generated = torch.full((b, 1), start_idx, dtype=torch.long, device=device)
        finished = torch.zeros(b, dtype=torch.bool, device=device)

        for _ in range(max_len - 1):
            logits = self.decoder(generated, memory)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            finished |= next_token.squeeze(1) == end_idx
            if finished.all():
                break

        return generated


## 6. Metrics (BLEU & ROUGE-L)

In [ ]:
# 6. NLG Evaluation Metrics
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu

def _lcs_length(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l(reference_tokens, hypothesis_tokens, beta: float = 1.2):
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    lcs = _lcs_length(reference_tokens, hypothesis_tokens)
    precision = lcs / len(hypothesis_tokens)
    recall = lcs / len(reference_tokens)
    if precision + recall == 0:
        return 0.0
    return ((1 + beta**2) * precision * recall) / (recall + beta**2 * precision)


## 7. Hyperparameters & DataLoader Setup
Regularized configuration with differential learning rates, weight decay, label smoothing, and dropout.

In [ ]:
# 7. Training Configuration
CONFIG = {
    "epochs": 35,
    "batch_size": 16,
    "lr": 3e-4,                  # Decoder & projection learning rate
    "backbone_lr": 5e-5,         # Lower learning rate for pretrained ResNet backbone
    "weight_decay": 1e-4,        # L2 Regularization
    "dropout": 0.2,              # Transformer dropout
    "patience": 6,               # Early stopping patience
    "image_size": 224,
    "max_len": 100,
    "max_images": 2,
    "d_model": 256,
    "nhead": 8,
    "num_layers": 4,
    "backbone": "resnet50",
    "pretrained": True,
    "min_word_freq": 3,
    "num_workers": 0,            # Avoids Python 3.12 multiprocessing issues
    "checkpoint_dir": "/kaggle/working/checkpoints",
    "vocab_path": "/kaggle/working/vocab.json"
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

# Build Vocabulary from training split
with open(annotation_path) as f:
    ann = json.load(f)

reports = [item["report"] for item in ann["train"]]
vocab = Vocabulary(min_freq=CONFIG["min_word_freq"]).build(reports)
vocab.save(CONFIG["vocab_path"])
print(f"Built vocabulary with {len(vocab)} tokens from {len(reports)} training reports.")
print(f"Saved vocabulary to {CONFIG['vocab_path']}")

# Instantiate Datasets & DataLoaders
train_ds = IUXrayDataset(
    annotation_path, image_dir, vocab, split="train",
    image_size=CONFIG["image_size"], max_len=CONFIG["max_len"], max_images=CONFIG["max_images"]
)
val_ds = IUXrayDataset(
    annotation_path, image_dir, vocab, split="val",
    image_size=CONFIG["image_size"], max_len=CONFIG["max_len"], max_images=CONFIG["max_images"]
)
test_ds = IUXrayDataset(
    annotation_path, image_dir, vocab, split="test",
    image_size=CONFIG["image_size"], max_len=CONFIG["max_len"], max_images=CONFIG["max_images"]
)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=(device.type == "cuda")
)

print(f"Dataset: {len(train_ds)} train | {len(val_ds)} val | {len(test_ds)} test samples")


## 8. Training Loop with Model Checkpointing & Early Stopping
Trains with AdamW (differential learning rate), Label Smoothing, LR Plateau decay, and saves the best model based on validation loss.

In [ ]:
# 8. Training Execution
model = ReportGenerator(
    vocab_size=len(vocab),
    d_model=CONFIG["d_model"],
    backbone=CONFIG["backbone"],
    pretrained=CONFIG["pretrained"],
    nhead=CONFIG["nhead"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    max_len=CONFIG["max_len"],
    pad_idx=vocab.word2idx[PAD_TOKEN],
).to(device)

# Cross-Entropy with Label Smoothing to prevent overconfident token memorization
criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx[PAD_TOKEN], label_smoothing=0.1)

# Differential Learning Rate: fine-tune backbone gently while training decoder actively
optimizer = torch.optim.AdamW([
    {"params": model.encoder.backbone.parameters(), "lr": CONFIG["backbone_lr"], "weight_decay": CONFIG["weight_decay"]},
    {"params": model.encoder.project.parameters(), "lr": CONFIG["lr"], "weight_decay": CONFIG["weight_decay"]},
    {"params": model.decoder.parameters(), "lr": CONFIG["lr"], "weight_decay": CONFIG["weight_decay"]},
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
epochs_no_improve = 0
best_model_path = os.path.join(CONFIG["checkpoint_dir"], "best_model.pt")

history = {"train_loss": [], "val_loss": []}
print(f"Starting training for up to {CONFIG['epochs']} epochs on {device}...")

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss = 0.0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [Train]")
    
    for images, reports in train_bar:
        images, reports = images.to(device, non_blocking=True), reports.to(device, non_blocking=True)
        input_ids = reports[:, :-1]
        target_ids = reports[:, 1:]

        optimizer.zero_grad()
        logits = model(images, input_ids)
        loss = criterion(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_ds)

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, reports in val_loader:
            images, reports = images.to(device, non_blocking=True), reports.to(device, non_blocking=True)
            input_ids = reports[:, :-1]
            target_ids = reports[:, 1:]
            logits = model(images, input_ids)
            loss = criterion(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
            val_loss += loss.item() * images.size(0)
            
    val_loss /= len(val_ds)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "args": CONFIG,
        }, best_model_path)
        print(f"  --> New best model saved (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CONFIG["patience"]:
            print(f"\nEarly stopping triggered at epoch {epoch} (no validation improvement for {CONFIG['patience']} epochs).")
            break

print(f"\nTraining complete! Best Validation Loss: {best_val_loss:.4f}")


## 9. Evaluation on Test Set
Evaluates the best checkpoint using greedy decoding and computes **BLEU-1**, **BLEU-4**, and **ROUGE-L**.

In [ ]:
# 9. Evaluate Best Checkpoint
print(f"Loading weights from {best_model_path}...")
ckpt = torch.load(best_model_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

references = []
hypotheses = []
rouge_scores = []
num_samples_to_print = 6
printed = 0

for images, reports in tqdm(test_loader, desc="Generating Test Reports"):
    images = images.to(device)
    generated_ids = model.generate(images, vocab, max_len=CONFIG["max_len"], device=device)

    for i in range(images.size(0)):
        hyp = vocab.decode(generated_ids[i].tolist())
        ref = vocab.decode(reports[i].tolist())
        hyp_tokens = hyp.split()
        ref_tokens = ref.split()

        hypotheses.append(hyp_tokens)
        references.append([ref_tokens])
        rouge_scores.append(rouge_l(ref_tokens, hyp_tokens))

        if printed < num_samples_to_print:
            print("-" * 65)
            print(f"[Sample {printed+1}]")
            print("REFERENCE :", ref)
            print("GENERATED :", hyp)
            printed += 1

print("\n" + "=" * 65)
print("TEST RESULTS")
print("=" * 65)
avg_rouge = sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0.0
smoothie = SmoothingFunction().method4
bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothie)
bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

print(f"Mean ROUGE-L : {avg_rouge:.4f}")
print(f"BLEU-1       : {bleu1:.4f}")
print(f"BLEU-4       : {bleu4:.4f}")
print("=" * 65)


## 10. Package Checkpoints for Local Deployment
Zips `best_model.pt` and `vocab.json` into `radiology_model_artifacts.zip` so you can download them and use them with your local Streamlit app and evaluation scripts.

In [ ]:
# 10. Zip and Prepare Download
import zipfile
from IPython.display import FileLink

zip_path = "/kaggle/working/radiology_model_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_model_path, arcname="best_model.pt")
    zf.write(CONFIG["vocab_path"], arcname="vocab.json")

print("Artifacts packaged successfully!")
print(f"Created: {zip_path}")
print(f"  - best_model.pt ({os.path.getsize(best_model_path)/(1024*1024):.2f} MB)")
print(f"  - vocab.json ({os.path.getsize(CONFIG['vocab_path'])/1024:.2f} KB)")
print("\nDownload link:")
FileLink(r'radiology_model_artifacts.zip')
